# 05 - Design de Sistemas
> Gateway, cache, retry e padroes de producao

**Modulo:** `08_arquitetura` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import time, hashlib

class LLMGateway:
    def __init__(self,client_,ttl=300,retries=3):
        self.c=client_; self._cache={}; self.ttl=ttl; self.retries=retries
        self._stats={'chamadas':0,'hits':0,'erros':0}

    def _key(self,p,s,m): return hashlib.md5(f'{p}{s}{m}'.encode()).hexdigest()

    def ask(self,prompt,system='',model=None,cache=True):
        model=model or HAIKU
        self._stats['chamadas']+=1
        k=self._key(prompt,system,model)
        if cache and k in self._cache:
            v,ts=self._cache[k]
            if time.time()-ts<self.ttl:
                self._stats['hits']+=1; return v
        for t in range(self.retries):
            try:
                r=self.c.messages.create(model=model,max_tokens=256,system=system,
                    messages=[{'role':'user','content':prompt}])
                txt=r.content[0].text
                self._cache[k]=(txt,time.time()); return txt
            except Exception as e:
                self._stats['erros']+=1
                if t<self.retries-1: time.sleep(2**t)
                else: raise

    def stats(self):
        h=self._stats['hits']; c=self._stats['chamadas']
        return {**self._stats,'hit_rate':f'{h/max(1,c):.0%}'}

gw=LLMGateway(client)
gw.ask('O que e Python?')
gw.ask('O que e Python?')  # cache hit
gw.ask('O que e ML?')
import json; print(json.dumps(gw.stats(),indent=2))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
